In [1]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD.parquet"
data: pd.DataFrame = read_data.read_asset(nombre_activo)

primer_lunes = data[data.index.dayofweek == 0].index[0]

data = data[primer_lunes:]

In [31]:
import find_best
from read_data import ohlc_form
from use_tecnics import main, simple_methods
import keys

keys.methods = simple_methods
keys.candles = 20

comp_sem = pd.Timedelta(days=4)
next_lun = pd.Timedelta(days=3)
ventana_ent = pd.Timedelta(weeks=3) + comp_sem

inicio = data.index[0].date()
fin = data.index[-1].date()

signals_and_prices = None

while True:
    train_time_init = inicio
    train_time_fin = inicio + ventana_ent

    test_time_init = train_time_fin + next_lun
    test_time_fin = test_time_init + comp_sem

    inicio += comp_sem + next_lun
    if inicio >= fin:
        break

    print(f"Etapa de entrenamiento desde {train_time_init} hasta  {train_time_fin}")
    data_w = data.loc[train_time_init: train_time_fin]
    best_ma = find_best.opti_main(data_w)

    data_w = data.loc[train_time_fin - comp_sem: test_time_fin]
    data_w = ohlc_form(data_w, f"{best_ma[1]}min")["close"]

    data_w = main(best_ma[0], data_w, best_ma[2:])
    data_w = data_w.loc[test_time_init: test_time_fin]

    f_index = data_w.index[0]
    l_index = data_w.index[-1]

    if data_w["Signals"][f_index] == -1:
        f_index = data_w["Signals"].index[1]

    if data_w["Signals"][l_index] == 1:
        l_index = data_w["Signals"].index[-2]

    data_w = data_w.loc[f_index: l_index]
    print(f"Guardando etapa de testeo desde {test_time_init} hasta  {test_time_fin}")
    if signals_and_prices is None:
        signals_and_prices = data_w
    else:
        signals_and_prices = pd.concat([signals_and_prices, data_w])

Etapa de entrenamiento desde 2025-02-24 hasta  2025-03-21
Resultado obtenido entrenando desde 2025-02-24 hasta 2025-03-20
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.7580645161290323
risk reward: 0.5694704136459483
profit factor: 2.4331917673963246
trades: 62
Resultado de sobre ajuste 1.0276106855268206
Guardando etapa de testeo desde 2025-03-24 hasta  2025-03-28
Etapa de entrenamiento desde 2025-03-03 hasta  2025-03-28
Resultado obtenido entrenando desde 2025-03-03 hasta 2025-03-27
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.746031746031746
risk reward: 0.6113518030468577
profit factor: 2.210271903323255
trades: 63
Resultado de sobre ajuste 1.0527502225600693
Guardando etapa de testeo desde 2025-03-31 hasta  2025-04-04
Etapa de entrenamiento desde 2025-03-10 hasta  2025-04-04
Resultado obtenido entrenando desde 2025-03-10 hasta 2025-04-03
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0

IndexError: index 0 is out of bounds for axis 0 with size 0

In [32]:
signals_and_prices.to_parquet("Data/señales_precios.parquet")

In [35]:
signals_and_prices

,Signals,Prices
time,,
2025-03-24 01:20:00,1,1.081920
2025-03-24 01:52:00,-1,1.082010
2025-03-24 04:32:00,1,1.082350
2025-03-24 04:48:00,-1,1.083050
2025-03-24 05:36:00,1,1.082640
...,...,...
2026-02-18 13:36:00,-1,1.183720
2026-02-18 13:52:00,1,1.183480
2026-02-18 14:08:00,-1,1.184055


In [33]:
from tester import backtest

backtest(signals_and_prices, False)

(np.float64(0.8229665071770335),
 np.float64(0.2935936585528526),
 np.float64(1.402725257530296),
 627)

In [49]:
initial_mon = 1000.0
cantidad = 0
apalancamiento = 10 

spread_pips = 0.0001

capital_maximo = initial_mon
drawdown_maximo = 0.0

for index in signals_and_prices.index:
    precio_actual = signals_and_prices["Prices"][index]
    current_signal = signals_and_prices["Signals"][index]

    if current_signal == 1 and cantidad == 0:
        poder_adquisitivo = initial_mon * apalancamiento
        
        precio_con_spread = precio_actual + spread_pips
        
        cantidad = int(poder_adquisitivo / precio_con_spread)

        if cantidad > 0:
            initial_mon -= (precio_con_spread * cantidad) 

    elif current_signal == -1 and cantidad > 0:
        initial_mon += (precio_actual * cantidad) 
        cantidad = 0

    equidad_actual = initial_mon + (precio_actual * cantidad) if cantidad > 0 else initial_mon
    
    if equidad_actual > capital_maximo:
        capital_maximo = equidad_actual
        
    drawdown_actual = ((capital_maximo - equidad_actual) / capital_maximo) * 100
    
    if drawdown_actual > drawdown_maximo:
        drawdown_maximo = drawdown_actual

print(f"Capital final CON SPREAD: ${initial_mon:.2f}")
print(f"Ganancia Neta: ${(initial_mon - 1000):.2f}")
print(f"Max Drawdown: {drawdown_maximo:.2f}%")

Capital final CON SPREAD: $1243.29
Ganancia Neta: $243.29
Max Drawdown: 27.90%
